# 54 — Human Review & Audit

Materializa la cola Top-K para revisión humana y preserva la separación entre evidencia del modelo y decisiones humanas. No genera ninguna sanción o conclusión jurídica.

In [1]:
from pathlib import Path

from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m5_gate = paths.docs / "decisions" / "M5_GATE.md"
assert m5_gate.exists() and "M5 ✅ CLOSED" in m5_gate.read_text(encoding="utf-8"), (
    "M5 no está formalmente cerrado. No iniciar M6."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
TEMP_DIR = paths.root / ".cache" / "duckdb"
SHADOW_INPUT = paths.root / "data" / "m5" / "fresh_oot_2026.parquet"

assert GOLD_SPEC.schema_version == M6_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M6_SPEC.core.benchmark_model_schema

{
    "m6_schema": M6_SPEC.schema_version,
    "model_version": M6_SPEC.core.model_version,
    "operating_mode": M6_SPEC.core.operating_mode,
    "scoring_mode": M6_SPEC.core.scoring_mode,
    "human_review_required": M6_SPEC.core.human_review_required,
    "production_promotion_authorized": M6_SPEC.deployment.allow_production_promotion,
}


{'m6_schema': 'm6-production-v1',
 'model_version': 'argos-v1',
 'operating_mode': 'shadow',
 'scoring_mode': 'batch',
 'human_review_required': True,
 'production_promotion_authorized': False}

In [2]:
from argos.production.scoring import score_batch
from argos.production.review import BUILD_IMPLEMENTATION, build_review_queue
from argos.production.audit import verify_audit_chain

scored = score_batch(
    input_path=SHADOW_INPUT, root=paths.root, gold_spec=GOLD_SPEC, modeling_spec=MODEL_SPEC,
    production_spec=M6_SPEC, reports_root=paths.reports, manifest_dir=MANIFEST_DIR, temp_directory=TEMP_DIR,
)
print("implementation:", BUILD_IMPLEMENTATION)
result = build_review_queue(
    scored_path=scored.scored_path,
    root=paths.root,
    production_spec=M6_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    temp_directory=TEMP_DIR,
)
result.queue_summary

implementation: m6-54-human-review-audit-v1


,batch_id,rows,distinct_contracts,min_rank,max_rank,review_status
0,a8865bfed9374218183eb302,5291,5291,1,5291,pending


In [3]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,review_queue_nonempty,human_review,True,True,"{""batch_id"": ""a8865bfed9374218183eb302"", ""dist...",operational top-K ranking materializes a uniqu...
1,review_queue_separates_model_and_human_fields,governance,True,True,"{""columns"": [""contract_id"", ""model_rank"", ""mod...",model evidence remains distinct from subsequen...
2,review_queue_initial_state,workflow,True,True,"{""allowed_statuses"": [""pending"", ""in_review"", ...",model prioritization creates pending reviews; ...
3,review_queue_privacy_minimized,privacy,True,True,"{""sensitive_present"": []}","review queue carries only identifiers, scores,..."


In [4]:
chain_ok, events, error = verify_audit_chain(paths.root, log_path=M6_SPEC.audit.log_path)
print({"audit_chain_ok": chain_ok, "events": events, "error": error})
assert chain_ok

{'audit_chain_ok': True, 'events': 15, 'error': None}


In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
print("54 ✅ PASS" if result.ready_for_next_notebook else "54 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Review queue:", result.queue_path)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook

54 ✅ PASS
Blocking checks: 4/4
Review queue: /Users/macm1pro/Desktop/projects/argos/argos-code/data/production/review_queues/a8865bfed9374218183eb302.parquet
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m6_human_review_queue.json
